# 海之子 · PPE 检测模型训练（Colab GPU）

本笔记本在 Google Colab 的免费 T4 GPU 上训练一个 YOLOv8n 的 PPE 检测模型，
导出 ONNX 后下载回本地，放进 `hzz-fire-safety/data/models/ppe_yolov8.onnx` 即可启用「施工 PPE / 危险检测」场景。

## 使用步骤
1. 打开 https://colab.research.google.com → 菜单 `文件 → 上传笔记本`，选择本文件。
2. 菜单 `运行时 → 更改运行时类型 → 硬件加速器 = GPU`（T4）。
3. 依次 `运行时 → 全部运行`（或逐格 Shift+Enter）。
4. 最后一个单元格会从浏览器下载 `best.onnx`，重命名为 `ppe_yolov8.onnx` 放到上面路径。

> 数据集：`LibreYOLO/construction-safety-gsnvb`（1206 张，CC-BY-4.0），
> 5 类：helmet / no-helmet / no-vest / person / vest。已与本地 `config.yaml` 的 class_map 对齐。

In [ ]:
# ① 安装依赖
!pip install -q ultralytics huggingface_hub

In [ ]:
# ② 下载训练集（自动拉取 construction-safety-gsnvb，约 75MB）
from huggingface_hub import snapshot_download

DATA_DIR = snapshot_download(
    repo_id="LibreYOLO/construction-safety-gsnvb",
    repo_type="dataset",
    local_dir="/content/ppe_dataset",
)
print("dataset:", DATA_DIR)

In [ ]:
# ③ 训练（GPU device=0；yolov8n 在 T4 上 50 epoch 约 3~6 分钟）
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # 首次自动下载基类权重
results = model.train(
    data="/content/ppe_dataset/data.yaml",
    epochs=50,
    imgsz=416,
    batch=16,
    device=0,            # 使用 Colab GPU
    project="/content/runs",
    name="ppe",
    verbose=True,
)
print("done")

In [ ]:
# ④ 导出 ONNX（opset 17，与本地 onnxruntime 兼容）
BEST = "/content/runs/ppe/weights/best.pt"
onnx_path = YOLO(BEST).export(format="onnx", imgsz=416, opset=17)
print("exported:", onnx_path)

In [ ]:
# ⑤ 从浏览器下载 ONNX（下载后重命名为 ppe_yolov8.onnx）
from google.colab import files
files.download("/content/runs/ppe/weights/best.onnx")